# 15. SFT and Alignment Basics

## Problem

为什么一个预训练模型还不等于一个好用的助手？SFT 到底改了什么，它和预训练明明都在算 cross entropy，为什么结果却会非常不同？

## Dependencies

建议先理解 pretraining、teacher forcing、token-level loss，以及 prompt-response 数据的基本结构。


## What SFT changes

预训练模型擅长的是“续写最像训练分布的文本”。这和“按用户要求回答问题”不是一回事。

比如同样一个问题，预训练模型可能会：

- 延续某种百科文风
- 模仿论坛语气
- 接着写一段常见模板
- 输出内容看似通顺，但没有真正完成任务

SFT 的作用，就是用更明确的示范数据把输出分布往“任务助手”方向拉过去。

它在做的事情不是重学语言，而是告诉模型：

- 当输入是一条指令时，回答应该如何组织
- 哪些信息应该优先出现
- 什么语气、格式、步骤、边界更合适
- 多轮对话里助手应该如何延续角色

所以可以把 SFT 看成：**在 base model 已经会说话的前提下，继续给它安装一个更清晰的行为骨架。**


## Core math

SFT 在形式上通常仍然使用 token-level cross entropy，但它并不是对整段文本一视同仁地算损失。常见做法是只对 assistant 输出部分计损失。

可以写成：

$$
L_{SFT} = -\frac{1}{\sum_t m_t} \sum_t m_t \log p_\theta(y_t \mid x, y_{<t})
$$

这里：

- $x$ 表示用户输入、系统提示词、上下文等条件信息。
- $y_t$ 表示 assistant 回答中的第 $t$ 个 token。
- $m_t$ 是 loss mask。
- 如果某个位置属于用户提示部分，常见地令 $m_t = 0$。
- 如果某个位置属于 assistant 回答部分，常见地令 $m_t = 1$。

这条公式非常重要，因为它说明：SFT 并不是单纯把所有 token 都平均对待，而是在告诉模型“哪些位置是需要学会生成的目标行为”。


In [ ]:
# 一个最小 SFT 样本。重点不是复杂 tokenizer，而是先看清楚数据格式。

sample = {
    'system': '你是一个简洁但准确的技术助手。',
    'user': '请用一句话解释什么是 BPE。',
    'assistant': 'BPE 是一种通过反复合并高频符号对来构建子词词表的分词方法。'
}

formatted_lines = [
    '<system>',
    sample['system'],
    '<user>',
    sample['user'],
    '<assistant>',
    sample['assistant'],
]

formatted_text = '
'.join(formatted_lines)
print(formatted_text)


## Why assistant-only loss is common

如果把用户输入部分也当成模型要预测的目标，那么模型会浪费大量容量去“复读提示词格式”，而不是把重点放在“如何生成合适回答”上。

所以很多 SFT 实现都会：

- 让模型把整段上下文都看进去
- 但只对 assistant token 计算损失

这样做的好处是，模型仍然知道完整对话条件是什么，但梯度主要落在“回答行为”上。

在多轮对话里，这种 mask 也可以扩展：

- user turn 的 token 不计损失
- assistant turn 的 token 计损失
- 如果有多个 assistant 回合，就对多个回答段一起计损失


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

# 假设一条格式化后的样本被切成 8 个位置。
# 其中前 3 个位置属于提示词或用户输入，不作为监督目标。
loss_mask = np.array([0, 0, 0, 1, 1, 1, 1, 1])

targets = np.array([1, 2, 3, 4, 5, 6, 7, 8])
logits = np.array([
    [ 0.2,  0.1, -0.5,  0.3,  0.0, -0.2,  0.4, -0.3,  0.1,  0.2],
    [ 0.4, -0.2,  0.3,  0.0,  0.2,  0.1, -0.1,  0.5, -0.4,  0.0],
    [ 0.1,  0.3,  0.2, -0.2,  0.5,  0.0, -0.3,  0.1,  0.4, -0.1],
    [ 0.0,  0.1, -0.1,  0.6,  0.3, -0.2,  0.2,  0.0,  0.1, -0.4],
    [ 0.2, -0.3,  0.1,  0.4,  0.7, -0.1,  0.0, -0.2,  0.3,  0.2],
    [ 0.3,  0.0, -0.2,  0.1,  0.2,  0.8, -0.1,  0.0,  0.2, -0.3],
    [ 0.1,  0.2,  0.0, -0.1,  0.3,  0.2,  0.9, -0.2,  0.1,  0.0],
    [-0.1,  0.3,  0.2,  0.0,  0.1, -0.2,  0.4,  1.0,  0.0,  0.2],
])

# 沿最后一维做 softmax，得到每个位置对词表的概率分布。
shifted = logits - np.max(logits, axis=-1, keepdims=True)
probs = np.exp(shifted) / np.exp(shifted).sum(axis=-1, keepdims=True)

# 每个位置先算出自己的 token loss。
per_token_loss = -np.log(probs[np.arange(len(targets)), targets])

# 再用 mask 只保留 assistant 位置的损失。
masked_loss = (per_token_loss * loss_mask).sum() / loss_mask.sum()

print('per_token_loss =', per_token_loss)
print('loss_mask =', loss_mask)
print('masked loss =', masked_loss)


In [ ]:
# 多轮对话里，mask 的意义会更明显。
# 这里我们用字符标签模拟一条多轮样本的“角色流”。

roles = [
    'system', 'system',
    'user', 'user', 'user',
    'assistant', 'assistant',
    'user', 'user',
    'assistant', 'assistant', 'assistant',
]

# 只对 assistant token 计损失。
assistant_mask = np.array([1 if role == 'assistant' else 0 for role in roles])

print('roles =', roles)
print('assistant loss mask =', assistant_mask)
print('number of supervised positions =', assistant_mask.sum())


## Pretraining vs SFT

它们表面上都可能写成 cross entropy，但本质差别很大。

**Pretraining** 更像在问：

- “真实世界文本分布里，接下来最可能出现什么？”

**SFT** 更像在问：

- “在这种任务条件下，一个理想助手更应该怎么回答？”

所以真正变化的不只是公式表面，而是：

- 数据分布变了
- 目标行为变了
- loss mask 变了
- 角色结构变了

这也是为什么 SFT 通常能显著提升模型可用性。base model 可能已经懂很多东西，但没有被系统性地推向“按指令配合”的输出模式。

## Common mistakes

- 把 SFT 误解成和 pretraining 完全不同的机器。实际上它们常常共享类似的 token-level loss 外壳。
- 忽略数据格式，以为只要堆问答文本就行。实际格式会显著影响模型学到的对话习惯。
- 认为 SFT 已经足以完成全部对齐。它更像是先把回答行为拉正，而不是解决全部偏好与长期稳定性问题。
- 没有区分“语言能力”和“助手行为”。前者主要来自预训练，后者主要来自 SFT 和后续偏好优化。

## Summary

SFT 是把 base model 推向 assistant model 的关键阶段。它不是从头再造能力，而是在已有语言能力上，重新塑造回答分布、角色边界和任务导向行为。下一步再引入 reward model 和 preference optimization，重点就会从“会回答”继续走向“更符合偏好地回答”。
